# Intelli-Speech — AUDIT_V2_FOLLOWUP (Experiment 0b + typical controls)

**Part A (read-only):** full-scale/clipping evidence, transcription script composition,
difficulty-label investigation. Output: `Intelli-Speech-data/analysis_v2/audit_followup_<date>/`.
The original Experiment 0 results (`audit_2026-11-22/`) are not touched.

**Part B (bounded download):** speaker-balanced typical Telugu controls from regular Vaani
(≤40 speakers, ≤10 clips each, ≤400 clips). Output: `Intelli-Speech-data/typical_controls_v2/`.
Plan first (metadata only), then download only the selected rows, with a hard size cap.

**Before running:** accept the access conditions for BOTH
https://huggingface.co/datasets/ARTPARK-IISc/Vaani-Atypical-Speech-Corpus and
https://huggingface.co/datasets/ARTPARK-IISc/Vaani with your own account.

Console output uses pseudonymous speaker labels (S1…). **Do not commit this notebook with
outputs** (research-only data); results live on Drive.

In [4]:
# CELL 1 | HUGGING FACE LOGIN (your own account; token never printed or saved in the repo)
from huggingface_hub import notebook_login, whoami
try:
    print("Already logged in as:", whoami()["name"])
except Exception:
    notebook_login()

Already logged in as: nix-999


In [5]:
# CELL 2 | DRIVE MOUNT + REPOSITORY SYNC
import os, subprocess, sys
from pathlib import Path
REPO_DIR = Path("/content/Intelli-Speech")
if not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", "https://github.com/niharika9nx/Intelli-Speech.git", str(REPO_DIR)], check=True)
else:
    dirty = subprocess.run(["git", "-C", str(REPO_DIR), "status", "--short"], capture_output=True, text=True).stdout.strip()
    print("Local changes (not pulling):\n" + dirty) if dirty else subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], check=True)
os.chdir(REPO_DIR); sys.path.insert(0, str(REPO_DIR))
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())
for f in ["scripts/audit_v2_followup.py", "scripts/build_typical_controls.py"]:
    assert Path(f).is_file(), f"{f} missing: pull the latest code first."
DATA_ROOT = "/content/drive/MyDrive/Intelli-Speech-data"
HF_CACHE = f"{DATA_ROOT}/hf_cache"

1563dae Add follow-up audit, typical controls, Experiment 1 feature pipeline; update agent config


In [6]:
# CELL 3 | DEPENDENCIES
!pip install -q datasets pyarrow soundfile scipy huggingface_hub

In [7]:
# CELL 4 | PART A — FOLLOW-UP AUDIT (read-only)
!python scripts/audit_v2_followup.py --data-root "{DATA_ROOT}" --hf-cache "{HF_CACHE}"

Wrote /content/drive/.shortcut-targets-by-id/1W9xt6UZaltMCWYwwxy_cly8t0_nFeaNM/Intelli-Speech-data/analysis_v2/audit_followup_2026-11-22/summary.json


In [8]:
# CELL 5 | PART A — RESULTS (pseudonymous speakers)
import json
import pandas as pd
OUT = sorted(Path(DATA_ROOT, "analysis_v2").glob("audit_followup_*"))[-1]
S = json.loads((OUT / "summary.json").read_text())
MAP = json.loads((OUT / "speaker_pseudonyms.json").read_text())["mapping"] if (OUT / "speaker_pseudonyms.json").is_file() else {}
def anon(obj):
    if isinstance(obj, dict): return {MAP.get(k, k): anon(v) for k, v in obj.items()}
    if isinstance(obj, list): return [anon(v) for v in obj]
    return obj
print("Output:", OUT)
for part in ["atypical_integrity", "typical_integrity"]:
    print(f"\n=== A1 {part} ==="); print(json.dumps(anon(S.get(part)), indent=1))
print("\n=== A2 transcription script composition ===")
print(json.dumps(S.get("atypical_text"), indent=1, ensure_ascii=False))
print(json.dumps(S.get("typical_text"), indent=1, ensure_ascii=False))
d = S.get("difficulty", {})
print("\n=== A3 difficulty ===")
print("documentation lines (verbatim):"); [print("  ", l) for l in d.get("documentation_lines", [])]
print("levels:", d.get("levels")); print(d.get("direction_conclusion"))
rows = [{"feature": f, "pooled_cliffs_delta_M_vs_L": v["pooled_cliffs_delta_medium_vs_low"], **{f"within_{k}": x for k, x in v["within_speaker"].items()}} for f, v in d.get("features", {}).items()]
print(pd.DataFrame(rows).to_string(index=False))

Output: /content/drive/MyDrive/Intelli-Speech-data/analysis_v2/audit_followup_2026-11-22

=== A1 atypical_integrity ===
{
 "files": 189,
 "analysed": 189,
 "subtypes": {
  "PCM_16": 189
 },
 "verdicts": {
  "consistent_with_peak_normalisation": 172,
  "clipping_evidence": 17
 },
 "exact_full_scale_samples": {
  "count": 189,
  "mean": 2.0688,
  "median": 1.0,
  "min": 1.0,
  "max": 44.0
 },
 "near_full_scale_samples": {
  "count": 189,
  "mean": 2.2593,
  "median": 1.0,
  "min": 1.0,
  "max": 44.0
 },
 "near_full_scale_fraction": {
  "count": 189,
  "mean": 0.0,
  "median": 0.0,
  "min": 0.0,
  "max": 0.0001
 },
 "max_consecutive_near_full_scale": {
  "count": 189,
  "mean": 1.0952,
  "median": 1.0,
  "min": 1.0,
  "max": 5.0
 },
 "files_with_run_ge_3": 5,
 "verdicts_by_speaker": {
  "S1": {
   "consistent_with_peak_normalisation": 41,
   "clipping_evidence": 4
  },
  "S2": {
   "consistent_with_peak_normalisation": 40
  },
  "S3": {
   "consistent_with_peak_normalisation": 5
  },
  "S

In [10]:
# CELL 6 | PART B1 — CONTROL SELECTION PLAN (metadata only, no audio)
!python scripts/build_typical_controls.py --phase plan --data-root "{DATA_ROOT}" --max-shards 30
PLAN = json.loads(Path(DATA_ROOT, "typical_controls_v2", "plan.json").read_text())
for k in ["telugu_shards_total", "shards_scanned", "scanned_rows", "telugu_rows", "transcribed_rows", "speakers_seen",
          "eligible_speakers", "selected_speakers", "selected_rows", "shortfall", "selected_gender_speakers",
          "selected_districts", "estimated_audio_bytes"]:
    print(f"{k:28s} {PLAN.get(k)}")
print(f"estimated download: {PLAN['estimated_audio_bytes']/1e9:.2f} GB")

{
 "created_utc": "2026-11-22T08:43:54.910996+00:00",
 "source_repo": "ARTPARK-IISc/Vaani",
 "source_revision": "9e6ed4f94f7060e633cbc500480623ae7521522c",
 "telugu_shards_total": 872,
 "shards_scanned": 30,
 "estimated_audio_bytes": 1126158077,
 "scanned_rows": 51504,
 "telugu_rows": 51504,
 "duration_ok_rows": 48651,
 "transcribed_rows": 3334,
 "speakers_seen": 666,
 "eligible_speakers": 47,
 "selected_speakers": 40,
 "selected_rows": 126,
 "target_speakers": 40,
 "shortfall": false,
 "config": {
  "n_speakers": 40,
  "max_per_speaker": 10,
  "max_total": 400,
  "min_per_speaker": 3,
  "min_duration_s": 2.0,
  "max_duration_s": 30.0,
  "require_transcript": true,
  "balance_gender": true,
  "salt": "intelli-speech-typical-controls-v2"
 },
 "selected_gender_speakers": {
  "Female": 20,
  "Male": 20
 },
 "selected_districts": 10
}
telugu_shards_total          872
shards_scanned               30
scanned_rows                 51504
telugu_rows                  51504
transcribed_rows      

In [11]:
# CELL 7 | PART B2 — BOUNDED DOWNLOAD (only selected rows; hard size cap)
MAX_DOWNLOAD_GB = 3.0
if PLAN.get("selected_rows", 0) == 0:
    print("Nothing selected; not downloading. Report the plan to Claude.")
elif PLAN["estimated_audio_bytes"] / 1e9 > MAX_DOWNLOAD_GB:
    print(f"Estimated {PLAN['estimated_audio_bytes']/1e9:.2f} GB exceeds the {MAX_DOWNLOAD_GB} GB cap; not downloading. Report to Claude.")
else:
    !python scripts/build_typical_controls.py --phase download --data-root "{DATA_ROOT}"

'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/ARTPARK-IISc/Vaani/resolve/main/audio/Telugu/train-00000-of-00872.parquet
Retrying in 1s [Retry 1/5].
{
 "created_utc": "2026-11-22T08:46:31.756992+00:00",
 "source_repo": "ARTPARK-IISc/Vaani",
 "source_revision": "9e6ed4f94f7060e633cbc500480623ae7521522c",
 "license": "CC-BY-4.0 (gated; access conditions accepted by the downloading user)",
 "rows_planned": 126,
 "files_written": 126,
 "duplicates_skipped": 0,
 "decode_errors": 0,
 "speakers": 40,
 "clips_per_speaker": {
  "count": 40.0,
  "mean": 3.15,
  "std": 0.43,
  "min": 3.0,
  "25%": 3.0,
  "50%": 3.0,
  "75%": 3.0,
  "max": 5.0
 },
 "total_minutes": 12.43,
 "resampled_files": 0,
 "original_sample_rates": {
  "16000": 126
 }
}


In [12]:
# CELL 8 | PART B3 — CONTROL SET STATISTICS (pseudonymous speakers)
CTL = Path(DATA_ROOT, "typical_controls_v2")
if (CTL / "provenance.json").is_file():
    P = json.loads((CTL / "provenance.json").read_text()); print(json.dumps(P, indent=1))
    m = pd.read_csv(CTL / "manifest.csv", dtype={"speakerID": str})
    ok = m[m.status == "ok"]
    per = ok.groupby("speakerID").agg(clips=("control_id", "count"), minutes=("duration_s", lambda s: round(s.sum() / 60, 2)),
                                      gender=("gender", "first"), district=("district", "first")).reset_index(drop=True)
    per.index = [f"T{i+1}" for i in range(len(per))]
    print(per.to_string())
    print("status counts:", m.status.value_counts().to_dict())

{
 "created_utc": "2026-11-22T08:46:31.756992+00:00",
 "source_repo": "ARTPARK-IISc/Vaani",
 "source_revision": "9e6ed4f94f7060e633cbc500480623ae7521522c",
 "license": "CC-BY-4.0 (gated; access conditions accepted by the downloading user)",
 "rows_planned": 126,
 "files_written": 126,
 "duplicates_skipped": 0,
 "decode_errors": 0,
 "speakers": 40,
 "clips_per_speaker": {
  "count": 40.0,
  "mean": 3.15,
  "std": 0.43,
  "min": 3.0,
  "25%": 3.0,
  "50%": 3.0,
  "75%": 3.0,
  "max": 5.0
 },
 "total_minutes": 12.43,
 "resampled_files": 0,
 "original_sample_rates": {
  "16000": 126
 }
}
     clips  minutes  gender         district
T1       3     0.42  Female         Chittoor
T2       3     0.34  Female           Guntur
T3       3     0.33  Female           Guntur
T4       3     0.44  Female          Krishna
T5       3     0.34  Female          Krishna
T6       3     0.33  Female          Krishna
T7       3     0.32  Female          Krishna
T8       3     0.26  Female          Krishna
T9  

## After running
Save the notebook (Ctrl+S) and tell Claude. Do not commit it with outputs.